<a href="https://colab.research.google.com/github/avi-dot-ai/FL-W/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


**One model row = one pseudonymized content item for one pseudonymized client, summarized over March 2026.** The underlying daily fact has one row per `report_date × client_hash_id × content_hash_id`; I aggregate those daily records to the content-month decision grain.

**Tables:** I use `fact_content_daily_performance` only. March 1–31, 2026 is the feature window and April 1–30, 2026 is the later outcome window. March is deliberately a mid-panel month, rather than the June sample/final month.

**Prediction:** at the end of March, estimate each eligible page's impression-weighted Google Search Console average position in April. Lower position is better; the output is a review-priority signal for an SEO specialist, not an automated site change.

**Deliberately excluded:** client and content hashes are context for grouping and splitting, never model features. April measurements are excluded from the honest feature set because they occur after the March decision moment.

In [1]:
%pip -q install duckdb scikit-learn

In [2]:
import os
import getpass
import duckdb

try:
  from google.colab import userdata
  HF_TOKEN = userdata.get('HF_TOKEN')
except (ImportError, KeyError):
  HF_TOKEN = os.environ.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
APRIL = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features (all March-only):** `march_impressions`, `march_clicks`, `march_ctr`, `march_position_volatility`, and `march_active_days`.

**Label:** `april_weighted_position`, the April impression-weighted `gsc_avg_position`. This is an observed future proxy, not a claim that any March feature causes ranking movement.

**Context:** `client_hash_id` and `content_hash_id` identify the page/client grouping; `report_date` defines the windows. They are used for aggregation and client-held-out evaluation, not as model inputs.

**Excluded:** April impressions, clicks, position, and any April-derived values are future information. `ga4_sessions` is also excluded here: GA4 is zero-filled where `ga4_data_available IS FALSE`, and I am keeping this first feature frame to consistently observed March search signals.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
# Verification 1 — raw daily grain. Empty output means the declared daily key holds.
con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS rows_at_key
    FROM {MARCH}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,rows_at_key


In [5]:
# Verification 2 — size and date coverage of the March feature slice.
con.sql(f"""
    SELECT COUNT(*) AS rows_in_slice,
           MIN(report_date) AS first_date,
           MAX(report_date) AS last_date
    FROM {MARCH}
""").df()

,rows_in_slice,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [6]:
# Verification 3 — GA4 availability. FALSE is unavailable, not zero engagement.
con.sql(f"""
    SELECT COUNT(*) AS all_march_rows,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)
             AS rows_with_ga4_available
    FROM {MARCH}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,all_march_rows,rows_with_ga4_available
0,9841378,413966.0


### Five-feature frame

- `march_impressions` is knowable at the March 31 decision moment because it sums only March Search Console observations.
- `march_clicks` is knowable at the March 31 decision moment because it sums only March Search Console observations.
- `march_ctr` is knowable at the March 31 decision moment because it is calculated only from March clicks and impressions.
- `march_position_volatility` is knowable at the March 31 decision moment because it describes variation in March observed positions only.
- `march_active_days` is knowable at the March 31 decision moment because it counts observed March reporting dates only.

In [7]:
frame = con.sql(f"""
    WITH march_features AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(COALESCE(gsc_impressions, 0)) AS march_impressions,
            SUM(COALESCE(gsc_clicks, 0)) AS march_clicks,
            SUM(COALESCE(gsc_clicks, 0))
                / NULLIF(SUM(COALESCE(gsc_impressions, 0)), 0) AS march_ctr,
            STDDEV_SAMP(NULLIF(gsc_avg_position, 0)) AS march_position_volatility,
            COUNT(DISTINCT report_date) AS march_active_days
        FROM {MARCH}
        GROUP BY 1, 2
        HAVING SUM(COALESCE(gsc_impressions, 0)) >= 100
    ),
    april_outcome AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position * gsc_impressions END)
                / NULLIF(SUM(CASE WHEN gsc_avg_position > 0 THEN gsc_impressions END), 0)
                AS april_weighted_position
        FROM {APRIL}
        GROUP BY 1, 2
    )
    SELECT m.*, a.april_weighted_position
    FROM march_features AS m
    INNER JOIN april_outcome AS a USING (client_hash_id, content_hash_id)
""").df()

print(f'Eligible March content-month rows: {len(frame):,}')
print(f'Clients represented: {frame.client_hash_id.nunique():,}')


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Eligible March content-month rows: 101,441
Clients represented: 44


### The deliberate leakage trap

I first score April position honestly from the five March-only features using a client-held-out split. I then add `april_weighted_position` itself as a deliberately leaked column. Its near-perfect score is expected because it is the target, measured after the decision. I remove it and retain the honest MAE.

In [8]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupShuffleSplit

feature_cols = [
    'march_impressions', 'march_clicks', 'march_ctr',
    'march_position_volatility', 'march_active_days'
]
model_data = frame.dropna(subset=feature_cols + ['april_weighted_position']).copy()
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(
    model_data[feature_cols],
    model_data['april_weighted_position'],
    groups=model_data['client_hash_id']
))

def score_mae(columns):
    model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    model.fit(model_data.iloc[train_idx][columns], model_data.iloc[train_idx]['april_weighted_position'])
    prediction = model.predict(model_data.iloc[test_idx][columns])
    return mean_absolute_error(model_data.iloc[test_idx]['april_weighted_position'], prediction)

honest_mae = score_mae(feature_cols)
print(f'Honest five-feature client-held-out MAE: {honest_mae:.3f} position points')

leaked_feature_cols = feature_cols + ['april_weighted_position']
leaked_mae = score_mae(leaked_feature_cols)
print(f'Deliberately leaked MAE: {leaked_mae:.3f} position points')

# The leaked feature is removed. This is the only result retained for the honest model.
final_feature_cols = feature_cols
print('Final honest feature count:', len(final_feature_cols))

Honest five-feature client-held-out MAE: 8.771 position points
Deliberately leaked MAE: 0.003 position points
Final honest feature count: 5


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


This is an unbalanced panel: clients begin reporting at different dates, so a March feature history does not mean the same depth of historical coverage for every client. The daily facts also contain GSC-only early rows where `ga4_data_available IS FALSE`; GA4 zeros on those rows are unavailable measurements, not evidence of no engagement. Finally, this observed-position proxy supports a review queue only—it cannot show that changing a page will cause a Google ranking improvement.

## Self-check


- Every section above is filled — markdown thinking AND the code that backs it
- The notebook runs top to bottom with no errors (Runtime → Run all)
- No client names, URLs, or private queries anywhere
- My claims use careful words: observed, measured, directional, decision-support
- Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.